In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a beautiful, clean style for the graphs
sns.set_theme(style="whitegrid")

# 1. Load the dataset
df = pd.read_csv('FINAL_EDA_READY_DATASET.csv')

# Remove any rows where income is 0 or negative just for the graphing 
# (to avoid messing up our median calculations and ratios)
df_clean = df[df['Real_HH_Income_2023'] > 0].copy()

# ==========================================
# GRAPH 1: The Macro Trend Line (Dual Axis)
# ==========================================
# Calculate the median values for each year
yearly_macro = df_clean.groupby('Year')[['Real_Median_Home_Price_2023', 'Real_HH_Income_2023']].median().reset_index()

fig1, ax1 = plt.subplots(figsize=(12, 6))

# Plot Income on the left y-axis
color1 = 'tab:blue'
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Real Median Household Income (2023 $)', color=color1, fontsize=12)
ax1.plot(yearly_macro['Year'], yearly_macro['Real_HH_Income_2023'], color=color1, linewidth=3, marker='o', label='Income')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(yearly_macro['Year'])
ax1.set_xticklabels(yearly_macro['Year'], rotation=45)

# Create a second y-axis for Home Prices (since the numbers are so much bigger)
ax2 = ax1.twinx()  
color2 = 'tab:red'
ax2.set_ylabel('Real Median Home Price (2023 $)', color=color2, fontsize=12)
ax2.plot(yearly_macro['Year'], yearly_macro['Real_Median_Home_Price_2023'], color=color2, linewidth=3, marker='s', label='Home Price')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Macro Trend: Real Income vs. Real Home Price Growth (2005-2023)', fontsize=16, fontweight='bold')
fig1.tight_layout()
plt.savefig('Graph_1_Macro_Trend.png')
plt.show()

# ==========================================
# GRAPH 2: The Generational Gap
# ==========================================
# Create a Home-to-Income Ratio column (Price divided by Annual Income)
df_clean['Home_to_Income_Ratio'] = df_clean['Real_Median_Home_Price_2023'] / df_clean['Real_HH_Income_2023']

# Order the generations chronologically
gen_order = ['Silent Generation', 'Baby Boomers', 'Generation X', 'Millennials', 'Generation Z']

# Calculate the median ratio for each generation across the dataset
gen_ratio = df_clean.groupby('Generation')['Home_to_Income_Ratio'].median().reindex(gen_order).reset_index()
# Drop any generations that might have NaN (e.g., if we didn't have Silent Gen in the data)
gen_ratio = gen_ratio.dropna()

fig2, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=gen_ratio, x='Generation', y='Home_to_Income_Ratio', palette='viridis', ax=ax)

plt.title('The Generational Gap: Median Home-to-Income Ratio', fontsize=16, fontweight='bold')
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Years of Income Needed to Buy a Home', fontsize=12)

# Add the specific numbers on top of the bars for clarity
for i in ax.containers:
    ax.bar_label(i, fmt='%.1fx', padding=3, fontsize=11)

fig2.tight_layout()
plt.savefig('Graph_2_Generational_Gap.png')
plt.show()

# ==========================================
# GRAPH 3: The Breakeven Point (Rent vs Own)
# ==========================================
# Calculate the median monthly rent vs mortgage for each year
yearly_costs = df_clean.groupby('Year')[['Real_Monthly_Rent_Cost_2023', 'Real_Monthly_Mortgage_Cost_2023']].median().reset_index()

fig3, ax3 = plt.subplots(figsize=(12, 6))

plt.plot(yearly_costs['Year'], yearly_costs['Real_Monthly_Rent_Cost_2023'], color='purple', linewidth=3, marker='o', label='Median Monthly Rent')
plt.plot(yearly_costs['Year'], yearly_costs['Real_Monthly_Mortgage_Cost_2023'], color='orange', linewidth=3, marker='s', label='Median Monthly Mortgage')

plt.title('The Breakeven Point: Monthly Cost to Rent vs. Own (2023 $)', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Monthly Cost (2023 $)', fontsize=12)
plt.xticks(yearly_costs['Year'], rotation=45)

# Fill the gap between the lines to highlight the difference
plt.fill_between(yearly_costs['Year'], 
                 yearly_costs['Real_Monthly_Rent_Cost_2023'], 
                 yearly_costs['Real_Monthly_Mortgage_Cost_2023'], 
                 color='gray', alpha=0.2)

plt.legend(fontsize=12)
fig3.tight_layout()
plt.savefig('Graph_3_Breakeven_Point.png')
plt.show()

print("✅ All 3 graphs generated and saved successfully!")